In [ ]:

import csv

with open("appointments.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

print("Column names:")
print(reader.fieldnames)

print("\nTotal number of records:", len(rows))


FileNotFoundError: [Errno 2] No such file or directory: 'appointments.csv'

In [ ]:

import os
import csv

for filename in os.listdir():
    if filename.endswith(".csv"):
        with open(filename, "r", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            print("\nFile:", filename)
            print("Columns:", reader.fieldnames)



File: ecWvPsdV2FNmRZAa.csv
Columns: ['note_id', 'appointment_id', 'note_text']

File: jbFzPdd2gB2Ex2TY.csv
Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

File: qDCCmpyfss1kBJJB.csv
Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']


In [ ]:

import os

os.rename("qDCCmpyfss1kBJJB.csv", "appointments.csv")
os.rename("jbFzPdd2gB2Ex2TY.csv", "clinics.csv")
os.rename("ecWvPsdV2FNmRZAa.csv", "followup_notes.csv")

print("All three files renamed successfully!")


All three files renamed successfully!


In [ ]:

import csv

with open("appointments.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

print("Column names:")
print(reader.fieldnames)

print("\nTotal number of records:", len(rows))


Column names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

Total number of records: 1020


In [ ]:

def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

print("Function created successfully!")


Function created successfully!


In [ ]:

import json

accepted = []
rejected = []

for row in rows:
    try:
        parsed_row = parse_row(row)
        accepted.append(parsed_row)

    except ValueError as error:
        rejected.append({
            "row": row,
            "reason": str(error)
        })

print("Total records:", len(rows))
print("Accepted records:", len(accepted))
print("Rejected records:", len(rejected))

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Rejected records saved successfully!")


Total records: 1020
Accepted records: 988
Rejected records: 32
Rejected records saved successfully!


In [ ]:

import pandas as pd
import numpy as np

# Load the original appointments dataset
df = pd.read_csv("appointments.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


Dataset loaded successfully!
Dataset shape: (1020, 14)


In [ ]:

# Display dataset information
print("Dataset information:")
df.info()

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check exact duplicate rows
print("\nDuplicate rows:")
print(df.duplicated().sum())


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patient_id          0
clinic_id 

In [ ]:

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))


Rows after removing duplicates: 1000


In [ ]:

df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print("Booking channels cleaned successfully!")
print(df["booking_channel"].value_counts())
print("Rows after cleaning booking_channel:", len(df))

Booking channels cleaned successfully!
booking_channel
App        382
Phone      355
Walk-In    253
Name: count, dtype: int64
Rows after cleaning booking_channel: 990


In [ ]:

df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)

print("Date columns converted successfully!")
print(df[["booked_on", "appt_date"]].dtypes)
print("Rows after converting dates:", len(df))


Date columns converted successfully!
booked_on    datetime64[ns]
appt_date    datetime64[ns]
dtype: object
Rows after converting dates: 990


In [ ]:

df = df.dropna(subset=["appt_date", "status"])

print("Rows after removing missing appointment dates/status:")
print(len(df))


Rows after removing missing appointment dates/status:
990


In [ ]:
# Replace invalid ages with missing values
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

print("Invalid ages replaced with missing values.")
print("Missing ages:", df["age"].isnull().sum())

Invalid ages replaced with missing values.
Missing ages: 17


In [ ]:
# Fill missing age with median
df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)

# Fill missing distance with median
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

print("Missing values filled successfully!")
print("Missing ages:", df["age"].isnull().sum())
print("Missing distances:", df["distance_km"].isnull().sum())

Missing values filled successfully!
Missing ages: 0
Missing distances: 0


In [ ]:
# Calculate quartiles for distance_km
Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)
IQR_distance = Q3_distance - Q1_distance

# Calculate quartiles for fee_inr
Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)
IQR_fee = Q3_fee - Q1_fee

print("Distance Q1:", Q1_distance)
print("Distance Q3:", Q3_distance)
print("Distance IQR:", IQR_distance)

print("Fee Q1:", Q1_fee)
print("Fee Q3:", Q3_fee)
print("Fee IQR:", IQR_fee)

Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5


In [ ]:
# Calculate distance limits
lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

# Calculate fee limits
lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

# Count values that will be capped
distance_capped = ((df["distance_km"] < lower_distance) |
                   (df["distance_km"] > upper_distance)).sum()

fee_capped = ((df["fee_inr"] < lower_fee) |
              (df["fee_inr"] > upper_fee)).sum()

# Cap outliers
df["distance_km"] = np.clip(
    df["distance_km"], lower_distance, upper_distance
)

df["fee_inr"] = np.clip(
    df["fee_inr"], lower_fee, upper_fee
)

print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Distance values capped: 26
Fee values capped: 3


In [ ]:
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Rounding completed successfully!")
print(df[["age", "distance_km", "fee_inr"]].head())

Rounding completed successfully!
   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700


In [ ]:
print("Missing ages:", df["age"].isnull().sum())
print("Missing distances:", df["distance_km"].isnull().sum())

print("\nData types:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)

print("\nFinal 5 rows:")
print(df[["age", "distance_km", "fee_inr"]].tail())

print("\nDataset shape:", df.shape)

Missing ages: 0
Missing distances: 0

Data types:
age              int64
distance_km    float64
fee_inr          int64
dtype: object

Final 5 rows:
      age  distance_km  fee_inr
1015   20         13.0      700
1016   77         17.4      400
1017   58          3.2      850
1018   59         25.6      300
1019   38         10.8      550

Dataset shape: (990, 14)


In [ ]:

# Create lead_days feature
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# Create no_show feature
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("New features created successfully!")
print(df[["booked_on", "appt_date", "lead_days", "status", "no_show"]].head())


New features created successfully!
   booked_on  appt_date  lead_days    status  no_show
0 2025-01-24 2025-01-25          1   No-Show        1
1 2025-05-27 2025-06-06         10  Attended        0
2 2025-03-24 2025-03-30          6  Attended        0
4 2025-04-18 2025-04-22          4  Attended        0
5 2024-11-23 2025-01-02         40   No-Show        1


In [ ]:

# Select required columns and convert to NumPy array
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

print("NumPy array created successfully!")
print("Array shape:", X.shape)
print(X[:5])


NumPy array created successfully!
Array shape: (990, 4)
[[ 60.   18.  850.    1. ]
 [ 39.   12.5 350.   10. ]
 [ 63.    4.1 850.    6. ]
 [ 52.    7.5 750.    4. ]
 [ 64.   19.  700.   40. ]]


In [ ]:
# Mean of each column
column_means = X.mean(axis=0)

print("Mean of each column (age, distance_km, fee_inr, lead_days):")
print(column_means)

Mean of each column (age, distance_km, fee_inr, lead_days):
[ 48.17373737   9.50090909 635.71414141  11.81919192]


In [ ]:
# Count and share of appointments booked more than 14 days ahead
mask = df["lead_days"] > 14

print("Rows with lead_days > 14:", mask.sum())
print("Share of rows with lead_days > 14:", mask.mean())

Rows with lead_days > 14: 296
Share of rows with lead_days > 14: 0.298989898989899


In [ ]:
# Calculate z-scores for distance_km
x = df["distance_km"].to_numpy()

z = (x - np.mean(x)) / np.std(x)

# Count values where absolute z-score is greater than 2
print("Number of distance_km values with |z| > 2:", (np.abs(z) > 2).sum())

Number of distance_km values with |z| > 2: 53


In [ ]:
import pandas as pd
import requests

# Load the original clinics dataset
clinics = pd.read_csv("clinics.csv")

print("Clinics dataset shape:", clinics.shape)
print("\nColumn names:")
print(clinics.columns.tolist())

print("\nFirst 5 rows:")
print(clinics.head())

Clinics dataset shape: (8, 6)

Column names:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

First 5 rows:
  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873


In [ ]:
# Function to fetch temperature for one clinic
def get_temperature(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code != 200:
                reason = f"HTTP status code {response.status_code}"
                continue

            data = response.json()

            if ("hourly" not in data or
                "temperature_2m" not in data["hourly"] or
                not data["hourly"]["temperature_2m"]):
                reason = "Hourly temperature data missing"
                continue

            return data["hourly"]["temperature_2m"][0]

        except requests.exceptions.RequestException as e:
            reason = str(e)

    print("Failed:", reason)
    return None


# Fetch temperature for all 8 clinics
clinics["temp_c"] = clinics.apply(
    lambda row: get_temperature(row["latitude"], row["longitude"]),
    axis=1
)

print("\nEnriched clinics:")
print(clinics)


Enriched clinics:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    19.4  
1    19.7  
2    19.7  
3    16.4  
4    16.4  
5    26.3  
6    16.1  
7    14.7  


In [ ]:
clinics.to_csv("clinics_enriched.csv", index=False)

print("clinics_enriched.csv saved successfully!")
print("Final shape:", clinics.shape)
print(clinics[["clinic_name", "temp_c"]])

clinics_enriched.csv saved successfully!
Final shape: (8, 7)
                clinic_name  temp_c
0     Kothrud Family Clinic    19.4
1      Hadapsar Care Centre    19.7
2    Sitabuldi Health Point    19.7
3      Malviya Nagar Clinic    16.4
4     Vaishali Nagar Clinic    16.4
5  Edappally Medical Centre    26.3
6        Vijay Nagar Clinic    16.1
7    Gomti Nagar Care Point    14.7


In [ ]:
# Required columns in the exact order
required_columns = [
    "appointment_id", "patient_id", "clinic_id",
    "booked_on", "appt_date", "age", "gender",
    "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows",
    "status", "lead_days", "no_show"
]

# Create a clean copy
export_df = df[required_columns].copy()

# Check appointment_id uniqueness
print("Are appointment IDs unique?",
      export_df["appointment_id"].is_unique)

# Check missing values
print("\nMissing values in each column:")
print(export_df.isnull().sum())

# Check whether all columns have no missing values
print("\nAny missing values?",
      export_df.isnull().any().any())

# Export only if both checks pass
if (export_df["appointment_id"].is_unique
        and not export_df.isnull().any().any()):

    # Sort by appointment_id
    export_df = export_df.sort_values("appointment_id")

    # Format date columns
    export_df["booked_on"] = export_df["booked_on"].dt.strftime("%Y-%m-%d")
    export_df["appt_date"] = export_df["appt_date"].dt.strftime("%Y-%m-%d")

    # Save CSV
    export_df.to_csv("appointments_clean.csv", index=False)

    print("\napppointments_clean.csv saved successfully!")
    print("Final shape:", export_df.shape)
    print("\nFirst 5 rows:")
    print(export_df.head())

else:
    print("\nPlease fix the data quality issues before exporting.")

Are appointment IDs unique? True

Missing values in each column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Any missing values? False

apppointments_clean.csv saved successfully!
Final shape: (990, 16)

First 5 rows:
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
4          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
5          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_k

In [ ]:
# Load enriched clinics
clinics_df = pd.read_csv("clinics_enriched.csv")

# Open SQL file for writing
with open("load_data.sql", "w") as file:

    # First: Insert clinic records
    for _, row in clinics_df.iterrows():
        sql = (
            f"INSERT INTO clinics VALUES ("
            f"'{row['clinic_id']}', "
            f"'{row['clinic_name']}', "
            f"'{row['city']}', "
            f"'{row['tier']}', "
            f"{row['latitude']}, "
            f"{row['longitude']}, "
            f"{row['temp_c'] if pd.notna(row['temp_c']) else 'NULL'}"
            f");\n"
        )
        file.write(sql)

    # Second: Insert appointment records
    for _, row in export_df.iterrows():
        sql = (
            f"INSERT INTO appointments VALUES ("
            f"'{row['appointment_id']}', "
            f"'{row['patient_id']}', "
            f"'{row['clinic_id']}', "
            f"'{row['booked_on']}', "
            f"'{row['appt_date']}', "
            f"{row['age']}, "
            f"'{row['gender']}', "
            f"{row['distance_km']}, "
            f"{row['fee_inr']}, "
            f"'{row['booking_channel']}', "
            f"'{row['visit_type']}', "
            f"'{row['reminder_sent']}', "
            f"{row['prior_no_shows']}, "
            f"'{row['status']}', "
            f"{row['lead_days']}, "
            f"{row['no_show']}"
            f");\n"
        )
        file.write(sql)

print("load_data.sql created successfully!")
print("Clinic INSERT statements:", len(clinics_df))
print("Appointment INSERT statements:", len(export_df))

load_data.sql created successfully!
Clinic INSERT statements: 8
Appointment INSERT statements: 990


In [ ]:
# Read the generated SQL file
with open("load_data.sql", "r") as file:
    sql_lines = file.readlines()

print("Total SQL statements:", len(sql_lines))

print("\nFirst 2 statements (should be clinics):")
print("".join(sql_lines[:2]))

print("\nLast 2 statements (should be appointments):")
print("".join(sql_lines[-2:]))

# Verify statement order
first_appointment = next(
    i for i, line in enumerate(sql_lines)
    if line.startswith("INSERT INTO appointments")
)

print("All clinic statements come before appointments:",
      all(line.startswith("INSERT INTO clinics")
          for line in sql_lines[:first_appointment]))

Total SQL statements: 998

First 2 statements (should be clinics):
INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 19.4);
INSERT INTO clinics VALUES ('C02', 'Hadapsar Care Centre', 'Pune', 'Metro', 18.5089, 73.926, 19.7);


Last 2 statements (should be appointments):
INSERT INTO appointments VALUES ('A0999', 'P509', 'C05', '2025-03-30', '2025-04-12', 59, 'F', 25.6, 300, 'Walk-In', 'Follow-up', 'Yes', 0, 'Attended', 13, 0);
INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 550, 'App', 'Follow-up', 'Yes', 2, 'No-Show', 45, 1);

All clinic statements come before appointments: True


In [ ]:
df.to_csv('appointments_clean.csv', index=False)

from google.colab import files
files.download('appointments_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
clinics.to_csv('clinics_enriched.csv', index=False)

from google.colab import files
files.download('clinics_enriched.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>